# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Tweak `agent_configs` / `turns_per_agent` (and snapshot flags) to explore different models, prompts, and persistence paths.
- Run cells sequentially to execute Agora sessions, inspect histories, and optionally save/load snapshots.


In [ ]:
import sys
sys.path.append("../src")

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent, build_system_prompt
from agora.llm import OpenRouterClient
from agora.persistence import load_snapshot, save_snapshot


In [ ]:
def extract_instruction(cfg, key):
    entry = cfg.get(key)
    if entry is None:
        return None, True
    if isinstance(entry, str):
        return entry, True
    if isinstance(entry, dict):
        return entry.get('instruction'), bool(entry.get('keep', True))
    raise ValueError(f'Invalid entry for {key}: {entry}')


# Public debate
- Simplest possible 2-agent configuration


In [ ]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is trying to sell you a widget.'},
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]


In [ ]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        system_prompt = build_system_prompt(cfg, total_agents=len(agent_configs))
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=system_prompt,
            response_instruction=cfg['response_instruction'],
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent, verbose=True)
finally:
    client.close()


In [ ]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        elif turn.role in {'pre_interview', 'post_interview'}:
            label = 'pre-interview' if turn.role == 'pre_interview' else 'post-interview'
            print(f"Turn {turn.turn_id:02d} | {speaker} ({label}): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")


# Public debate with private reflections
- 3 agents example
- The option to save and/or load the state is also available

In [ ]:
# --- Private reflection configuration ---
private_turns_per_agent = 2
private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
            {'name': 'Gamma', 'role': 'Gamma is observing your conversation.'}
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response': {'instruction': 'Alpha, what do you think of this interaction so far?', 'keep': True},
        'pre_interview': {'instruction': 'Alpha, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Alpha, after this exchange, summarize how it went and any next steps.', 'keep': False},
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is your salesman.'},
            {'name': 'Gamma', 'role': 'Gamma is observing your conversation.'}
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response': {'instruction': 'Beta, what do you think of this interaction so far?', 'keep': True},
        'pre_interview': {'instruction': 'Beta, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Beta, after this exchange, summarize how it went and any next steps.', 'keep': False},
    },
    {
        'name': 'Gamma',
        'model': 'openai/gpt-4o-mini',
        'self_role': 'You are Gamma, listening to a conversation between a salesman and a potential buyer. You helpfully ad-lib their conversation in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is the salesman.'},
            {'name': 'Beta', 'role': 'Beta is the potential buyer.'}
        ],
        'response_instruction': 'Gamma, offer your next public utterance.',
        'private_response': {'instruction': 'Gamma, what do you think of this interaction so far?', 'keep': True},
        'pre_interview': {'instruction': 'Gamma, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Gamma, after this exchange, summarize how it went and any next steps.', 'keep': False},
    },
]

snapshot_path = Path('snapshots/reflection_snapshot.json')
load_snapshot_flag = False  # Set True to resume from a saved snapshot.
save_snapshot_flag = True  # Set True to persist the new session.


In [ ]:
# --- Run Agora with private reflections ---
private_agents = []
private_client = OpenRouterClient()
try:
    if load_snapshot_flag and snapshot_path.exists():
        def _snapshot_factory(state):
            return private_client
        private_agora = load_snapshot(snapshot_path, _snapshot_factory)
        private_agents = list(private_agora.agents)
        print(f'Loaded snapshot from {snapshot_path}')
    else:
        for cfg in private_agent_configs:
            system_prompt = build_system_prompt(cfg, total_agents=len(private_agent_configs))
            private_instr, private_keep = extract_instruction(cfg, 'private_response')
            pre_instr, pre_keep = extract_instruction(cfg, 'pre_interview')
            post_instr, post_keep = extract_instruction(cfg, 'post_interview')
            agent = Agent(
                name=cfg['name'],
                model=cfg['model'],
                llm_client=private_client,
                system_prompt=system_prompt,
                response_instruction=cfg['response_instruction'],
                private_response_instruction=private_instr,
                private_response_keep=private_keep,
                pre_interview_instruction=pre_instr,
                pre_interview_keep=pre_keep,
                post_interview_instruction=post_instr,
                post_interview_keep=post_keep,
            )
            private_agents.append(agent)
        private_agora = Agora(private_agents)

    private_history = private_agora.run(max_turns_per_agent=private_turns_per_agent, verbose=True)

    print(f'Total turns (including private reflections): {len(private_history)}')
    for turn in private_history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        note = ''
        if turn.role in {'reflection', 'pre_interview', 'post_interview'} and not turn.keep:
            note = ' (excluded)'
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private){note}: {turn.private_reflection}")
        elif turn.role in {'pre_interview', 'post_interview'}:
            label = 'pre-interview' if turn.role == 'pre_interview' else 'post-interview'
            print(f"Turn {turn.turn_id:02d} | {speaker} ({label}){note}: {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    private_client.close()


In [ ]:
# --- Save snapshot (optional) ---
if save_snapshot_flag:
    save_snapshot(snapshot_path, private_agora)
    print(f'Snapshot saved to {snapshot_path}')
else:
    print('Snapshot not saved (set save_snapshot_flag=True to persist).')


In [ ]:
# --- Inspect each agent's perspective on the history ---
for agent in private_agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        note = ''
        if turn.role in {'reflection', 'pre_interview', 'post_interview'} and not turn.keep:
            note = ' (excluded)'
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private){note}: {turn.private_reflection}")
        elif turn.role in {'pre_interview', 'post_interview'}:
            label = 'pre-interview' if turn.role == 'pre_interview' else 'post-interview'
            print(f"Turn {turn.turn_id:02d} | {speaker} ({label}){note}: {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
